In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tensorflow.keras.datasets import imdb

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
VOCAB_SIZE = 10000
MAX_LEN = 200

(x_train,y_train),(x_test,y_test)=imdb.load_data(num_words=VOCAB_SIZE)

def pad(x):
    return np.array([i[:MAX_LEN]+[0]*(MAX_LEN-len(i)) for i in x])

x_train=pad(x_train)
x_test=pad(x_test)

x_train=torch.tensor(x_train)
y_train=torch.tensor(y_train)

x_test=torch.tensor(x_test)
y_test=torch.tensor(y_test)

print(x_train.shape)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
torch.Size([25000, 200])


In [4]:
class BiLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB_SIZE,128)
        self.lstm = nn.LSTM(128,128,batch_first=True,bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(256,1)

    def forward(self,x):
        x = self.emb(x)
        out,_ = self.lstm(x)
        out = self.dropout(out[:,-1])
        return torch.sigmoid(self.fc(out))

In [10]:
model = BiLSTM().to(device)

loss_fn = nn.BCELoss()
opt = optim.Adam(model.parameters(),0.001)

for epoch in range(20):
    perm = torch.randperm(len(x_train))

    correct = 0
    total = 0

    for i in range(0,10000,64):   # subset for speed
        idx = perm[i:i+64]

        xb = x_train[idx].to(device)
        yb = y_train[idx].float().unsqueeze(1).to(device)

        opt.zero_grad()
        out = model(xb)
        loss = loss_fn(out,yb)
        loss.backward()
        opt.step()

        pred = (out>0.5).int()
        correct += (pred==yb.int()).sum().item()
        total += yb.size(0)

    print("Epoch",epoch+1,"Accuracy:",correct/total)

Epoch 1 Accuracy: 0.5025875796178344
Epoch 2 Accuracy: 0.5242834394904459
Epoch 3 Accuracy: 0.5329418789808917
Epoch 4 Accuracy: 0.5549363057324841
Epoch 5 Accuracy: 0.6132563694267515
Epoch 6 Accuracy: 0.6963574840764332
Epoch 7 Accuracy: 0.7455214968152867
Epoch 8 Accuracy: 0.573546974522293
Epoch 9 Accuracy: 0.6491839171974523
Epoch 10 Accuracy: 0.7708001592356688
Epoch 11 Accuracy: 0.8140923566878981
Epoch 12 Accuracy: 0.8193670382165605
Epoch 13 Accuracy: 0.8486265923566879
Epoch 14 Accuracy: 0.8741042993630573
Epoch 15 Accuracy: 0.8801751592356688
Epoch 16 Accuracy: 0.8967953821656051
Epoch 17 Accuracy: 0.9068471337579618
Epoch 18 Accuracy: 0.9183917197452229
Epoch 19 Accuracy: 0.9262539808917197
Epoch 20 Accuracy: 0.9320262738853503


In [11]:
with torch.no_grad():
    out = model(x_test[:2000].to(device))
    pred = (out>0.5).int().squeeze()
    acc = (pred.cpu()==y_test[:2000]).sum()/2000

    print("Test Accuracy:",acc.item())

Test Accuracy: 0.847000002861023


In [17]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.datasets import imdb

word_index = imdb.get_word_index()

def predict(text):
    tokens = text.lower().split()

    seq = []
    for w in tokens:
        idx = word_index.get(w)
        if idx is not None and idx < VOCAB_SIZE:
            seq.append(idx + 3)
        else:
            seq.append(2)

    seq = pad_sequences([seq], maxlen=MAX_LEN)

    t = torch.tensor(seq).to(device)
    out = model(t).item()

    print("Score:", out)
    print("Positive" if out > 0.5 else "Negative")

predict("this movie was amazing and emotional")
predict("worst film ever")

Score: 0.9813563823699951
Positive
Score: 0.1065717414021492
Negative
